# Feature Engineering


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, RobustScaler
import joblib
import sys
import os
from pathlib import Path

current_dir = Path().resolve()

if current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

sys.path.insert(0, str(project_root))
from src.data.data_loader import DataLoader

In [ ]:
class FeatureEngineer:
    def __init__(self):
        self.scaler = RobustScaler()  
        self.fitted = False
    
    def create_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        
        # 1. Amount-based features
        df['Amount_log'] = np.log1p(df['Amount'])
        df['Amount_squared'] = df['Amount'] ** 2
        df['Amount_sqrt'] = np.sqrt(df['Amount'])
        
        # 2. Time-based features
        # Convert seconds to hours
        df['Hour'] = (df['Time'] / 3600) % 24
        df['Day'] = (df['Time'] / 86400).astype(int)
        
        # Cyclical encoding for hour
        df['Hour_sin'] = np.sin(2 * np.pi * df['Hour'] / 24)
        df['Hour_cos'] = np.cos(2 * np.pi * df['Hour'] / 24)
        
        # Time segments
        df['Is_Night'] = ((df['Hour'] >= 0) & (df['Hour'] < 6)).astype(int)
        df['Is_Morning'] = ((df['Hour'] >= 6) & (df['Hour'] < 12)).astype(int)
        df['Is_Afternoon'] = ((df['Hour'] >= 12) & (df['Hour'] < 18)).astype(int)
        df['Is_Evening'] = ((df['Hour'] >= 18) & (df['Hour'] < 24)).astype(int)
        
        # 3. V features aggregations
        v_features = [col for col in df.columns if col.startswith('V')]
        df['V_sum'] = df[v_features].sum(axis=1)
        df['V_mean'] = df[v_features].mean(axis=1)
        df['V_std'] = df[v_features].std(axis=1)
        df['V_min'] = df[v_features].min(axis=1)
        df['V_max'] = df[v_features].max(axis=1)
        
        # 4. Interaction features (top correlated V features)
        df['V14_V12'] = df['V14'] * df['V12']
        df['V14_V10'] = df['V14'] * df['V10']
        df['V12_V10'] = df['V12'] * df['V10']
        
        # 5. Amount percentile within time window
        df = df.sort_values('Time')
        df['Amount_rolling_mean'] = df['Amount'].rolling(window=100, min_periods=1).mean()
        df['Amount_rolling_std'] = df['Amount'].rolling(window=100, min_periods=1).std()
        df['Amount_zscore'] = (df['Amount'] - df['Amount_rolling_mean']) / (df['Amount_rolling_std'] + 1e-5)
        
        return df
    
    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Fit scaler and transform features"""
        df_features = self.create_features(df)
        
        # Separate features and target
        if 'Class' in df_features.columns:
            y = df_features['Class']
            X = df_features.drop('Class', axis=1)
        else:
            y = None
            X = df_features
        
        # Scale features (except binary features)
        binary_cols = ['Is_Night', 'Is_Morning', 'Is_Afternoon', 'Is_Evening']
        cols_to_scale = [col for col in X.columns if col not in binary_cols]
        
        X[cols_to_scale] = self.scaler.fit_transform(X[cols_to_scale])
        self.fitted = True
        
        # Save feature names
        self.feature_names = X.columns.tolist()
        
        if y is not None:
            X['Class'] = y
        
        return X
    
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        if not self.fitted:
            raise ValueError("Scaler not fitted. Call fit_transform first.")
        
        df_features = self.create_features(df)
        
        if 'Class' in df_features.columns:
            y = df_features['Class']
            X = df_features.drop('Class', axis=1)
        else:
            y = None
            X = df_features
        
        binary_cols = ['Is_Night', 'Is_Morning', 'Is_Afternoon', 'Is_Evening']
        cols_to_scale = [col for col in X.columns if col not in binary_cols]
        
        X[cols_to_scale] = self.scaler.transform(X[cols_to_scale])
        
        if y is not None:
            X['Class'] = y
        
        return X
    
    def save(self, path: str = 'ml_models/saved_models/'):
        """Save scaler and feature names"""
        os.makedirs(path, exist_ok=True)
        joblib.dump(self.scaler, f'{path}/scaler.pkl')
        joblib.dump(self.feature_names, f'{path}/feature_names.pkl')
    
    def load(self, path: str = 'ml_models/saved_models/'):
        """Load scaler and feature names"""
        self.scaler = joblib.load(f'{path}/scaler.pkl')
        self.feature_names = joblib.load(f'{path}/feature_names.pkl')
        self.fitted = True

In [ ]:
try:
    data_path = None
    possible_paths = [
        'data/raw/creditcard.csv',  
        '../data/raw/creditcard.csv',  
        '../../data/raw/creditcard.csv'  
    ]
    
    for path in possible_paths:
        if os.path.exists(path):
            data_path = path
            break
    
    if data_path is None:
        raise FileNotFoundError(f"could not find creditcard.csv. Tried: {possible_paths}")
    
    print(f"Using data path: {data_path}")
    loader = DataLoader(data_path)
    
    print("Loading data...")
    df = loader.load_data()
    train, val, test = loader.create_splits(df)
    
    engineer = FeatureEngineer()
    
    print("Fitting and transforming train set...")
    X_train_processed = engineer.fit_transform(train)
    
    print("Transforming validation set...")
    X_val_processed = engineer.transform(val)
    
    print("Transforming test set...")
    X_test_processed = engineer.transform(test)
    
    print("\nFeature Engineering Complete!")
    print(f"Original Train Shape: {train.shape}")
    print(f"Processed Train Shape: {X_train_processed.shape}")
    
    
except Exception as e:
    print(f"An error occurred: {e}")
    import traceback
    traceback.print_exc()


Using data path: ../data/raw/creditcard.csv
Loading data...
loading data

Train set: 199364 (345 frauds)
Val set: 28481 (49 frauds)
Test set: 56962 (98 frauds)
Saving splits to: D:\Projects\Financial Fraud Detection System\data\processed
Fitting and transforming train set...
Transforming validation set...
Transforming test set...

Feature Engineering Complete!
Original Train Shape: (199364, 31)
Processed Train Shape: (199364, 53)
